# Создание моделей для предсказания свойств углепластика, полученного по вакуумной технологии

In [93]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import f_classif

import matplotlib.pyplot as plt 

# инструменты для построения модели:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # инструмент для создания и обучения модели
from sklearn.ensemble import RandomForestRegressor # инструмент для создания и обучения модели
from sklearn import metrics # инструменты для оценки точности модели
from xgboost import XGBRegressor

RANDOM_SEED = 42


In [94]:
df = pd.read_csv('data/dataset_prepaired.csv')
df.head()

,Linera density,Density yarn,Strength Gpa,Module Gpa,lengthening,Mass size,breaking the loop,Surface density of the fabric,Prepreg surface density,Resin content,viscosity,Gelation time,Resin Tg,technology,Thickness of the monolayer,density,Strength_plastik,Module_plastik,LSS,Plastik_Tg
0,188.0,1.758,4.59,253.0,1.814229,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
1,189.0,1.758,4.48,260.0,1.723077,1.1,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
2,188.0,1.759,4.28,257.0,1.665370,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
3,187.0,1.758,4.77,256.0,1.863281,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
4,190.0,1.757,4.56,255.0,1.788235,0.9,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0


In [95]:

df = df[df['technology'] == 1]
df = df.drop('technology', axis=1)

In [96]:
df = df.rename(columns={'Thickness of the monolayer' : 'Thickness_monolayer',
                        'Module_plastik ' : 'Module_plastik'})

### Модель для предсказания толщины монослоя Thickness_monolayer

In [97]:
train_data_thickness = df.drop(['density', 'Strength_plastik', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_thickness.drop(['Thickness_monolayer'], axis=1))
y = np.array(train_data_thickness.Thickness_monolayer.values)

In [98]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

In [99]:
# НАСТРОЙКИ 
model_rf_thikness = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

In [100]:
# обучаем модель на тестовом наборе данных
model_rf_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_thikness.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [101]:
def mean_absolute_percentage_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr) / y_tr)) * 100

In [102]:
# сравниваем предсказанные значения (y_pred) с реальными (y_test), 
# метрика mean squared error, MSE показывает среднеквадратичное отклонение:

def mean_squared_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr)**2)) 

In [103]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.011


In [104]:
model_lr_thikness = LinearRegression()

In [105]:
# обучаем модель на тестовом наборе данных
model_lr_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_thikness.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 1.389


### Модель для предсказания плотности углепластика density

In [106]:
train_data_density = df.drop(['Thickness_monolayer', 'Strength_plastik', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_density.drop(['density'], axis=1))
y = np.array(train_data_density.density.values)

In [107]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_density = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_density.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [108]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.001


In [109]:
model_lr_density = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_density.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.0
MAPE: 0.404


### Модель для предсказания прочности углепластика Strength

In [110]:
df['Strength Gpa'] = df['Strength Gpa'] * 1000
df

,Linera density,Density yarn,Strength Gpa,Module Gpa,lengthening,Mass size,breaking the loop,Surface density of the fabric,Prepreg surface density,Resin content,viscosity,Gelation time,Resin Tg,Thickness_monolayer,density,Strength_plastik,Module_plastik,LSS,Plastik_Tg
426,188.0,1.752,4290.0,254.0,1.688976,1.100000,26.20,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
427,184.0,1.751,4300.0,255.0,1.686275,1.200000,24.70,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
428,179.0,1.750,4620.0,268.0,1.723881,1.200000,26.30,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
429,189.0,1.749,4370.0,254.0,1.720472,1.100000,25.60,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
430,189.0,1.749,4260.0,255.0,1.670588,1.000000,24.80,197.5,321.370000,38.970000,25.20,19.00,152.00,0.209000,1.555000,922.0,71.70,78.9,156.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10706,183.0,1.763,4750.0,263.0,1.806084,1.175676,23.85,199.0,326.813333,39.043333,23.61,14.65,149.92,0.213667,1.525667,872.0,65.95,86.4,161.0
10707,183.0,1.764,4610.0,262.0,1.759542,1.175676,23.85,199.0,326.813333,39.043333,23.61,14.65,149.92,0.213667,1.525667,872.0,65.95,86.4,161.0
10708,186.0,1.763,4440.0,265.0,1.675472,1.175676,23.85,199.0,326.813333,39.043333,23.61,14.65,149.92,0.213667,1.525667,872.0,65.95,86.4,161.0
10709,184.0,1.763,4450.0,257.0,1.731518,1.175676,23.85,199.0,326.813333,39.043333,23.61,14.65,149.92,0.213667,1.525667,872.0,65.95,86.4,161.0


In [111]:
train_data_strength = df.drop(['Thickness_monolayer', 'Module_plastik',
       'LSS', 'Plastik_Tg', 'density'], axis=1)

X = np.array(train_data_strength.drop(['Strength_plastik'], axis=1))
y = np.array(train_data_strength.Strength_plastik.values)

In [112]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_strength = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_strength.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [113]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 19.13
MAPE: 0.032


In [114]:
model_lr_strength = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_strength.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 479.289
MAPE: 1.833


In [115]:
X = np.array([[190, 1.779, 4.6, 265, 1.7, 1.3, 20, 210, 333, 37.40, 32.33, 12.8, 149.0]])
model_lr_strength.predict(X)

array([1325.27583797])

In [116]:
model_rf_strength.predict(X)

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


array([1038.335])

In [117]:
model_xgb_strenght = XGBRegressor(lerning_rate = 0.01)

# обучаем модель на тестовом наборе данных
model_xgb_strenght.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_xgb_strenght.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 21.984
MAPE: 0.032


/home/alexandr/anaconda3/envs/ML/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:33:42] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "lerning_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [118]:
model_xgb_strenght.predict(X)

array([1050.4585], dtype=float32)

### Модель для предсказания модуля углепластика Module

In [119]:
train_data_module = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_module.drop(['Module_plastik'], axis=1))
y = np.array(train_data_module.Module_plastik .values)

In [120]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_module = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_module.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [121]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.018
MAPE: 0.011


In [122]:
model_lr_module = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_module.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 2.988
MAPE: 2.183


### Модель для предсказания межслоевой прочности углепластика LSS

In [123]:
train_data_lss = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik',
                             'Plastik_Tg'], axis=1)

X = np.array(train_data_lss.drop(['LSS'], axis=1))
y = np.array(train_data_lss.LSS.values)

In [124]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_lss = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_lss.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [125]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.156
MAPE: 0.032


In [126]:
model_lr_lss = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_lss.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 16.151
MAPE: 4.074


In [127]:
model_xgb_lss = XGBRegressor(lerning_rate = 0.01)

# обучаем модель на тестовом наборе данных
model_xgb_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_xgb_lss.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.155
MAPE: 0.026


/home/alexandr/anaconda3/envs/ML/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:33:43] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "lerning_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


# Модель для предсказания температуры стеклования углепластика Tg

In [128]:
train_data_Tg = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik', 
                         'LSS'], axis=1)

X = np.array(train_data_Tg.drop(['Plastik_Tg'], axis=1))
y = np.array(train_data_Tg.Plastik_Tg.values)

In [129]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_Tg = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_Tg.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [130]:
print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 0.018
MAPE: 0.005


In [131]:
model_lr_Tg = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_Tg.predict(X_test)

print('MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

MSE: 3.486
MAPE: 0.93
